# Downloading dataset from huggingface

In [ ]:
from datasets import load_dataset

words = load_dataset("swami93/indian-names-1.5M")
words.save_to_disk("/kaggle/working/indian_names")

In [8]:
from datasets import load_from_disk

dataset = load_from_disk("/kaggle/working/indian_names")

df = dataset["train"].to_pandas()

display(df.head())

,text
0,name: thiv
1,name: ziyan
2,name: del
3,name: bharathushan
4,name: aranamuthan


In [10]:
print(df.shape)
print(df.columns)

(114239, 1)
Index(['text'], dtype='object')


In [13]:
from datasets import load_from_disk

dataset = load_from_disk("/kaggle/working/indian_names")
names = dataset["train"]["text"]
names = [
    name.replace("name: ", "", 1).strip()
    for name in names
    if name
]
with open("/kaggle/working/indian_names.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(names))

print(f"Saved {len(names)} names")

Saved 114239 names


# Actual start

In [265]:
file_path = "/kaggle/input/datasets/yogeshvvvv/indian-names-sample/indian_names.txt"

with open(file_path, "r", encoding="utf-8") as f:
    words = [line.strip() for line in f if line.strip()]

print(words[:10])
print("Total names:", len(words))

['thiv', 'ziyan', 'del', 'bharathushan', 'aranamuthan', 'saptahotre', 'lakshay', 'mangaraj', 'gopitha', 'jeylani']
Total names: 114239


In [266]:
words = [w for w in words if len(w) >= 3]
#remove words whose len less than 3

In [269]:
#split each name into trigrams
b = {}
for word in words:
    chs = ['<S>'] + list(word) + ['<E>']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        trigram = ch1, ch2, ch3
        b[trigram] = b.get(bigram, 0)+1
#sorted(b.items(), key = lambda kv: -kv[1])

In [270]:
import torch

In [271]:
N = torch.zeros((27,27, 27), dtype=torch.int32)

In [272]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.']=0
itos = {i:s for s,i in stoi.items()}

In [274]:
#trigram-wise occurence frequency
for word in words:
    if word == '':
        continue
    chs = ['.'] + list(word) + ['.']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]
        N[ix1,ix2, ix3] += 1

In [275]:
#plot everything
import plotly.graph_objects as go
import numpy as np

x, y, z = np.indices(N.shape)

fig = go.Figure(data=[go.Scatter3d(
    x=x.flatten(),
    y=y.flatten(),
    z=z.flatten(),
    mode='markers',
    marker=dict(size=3,color=N.flatten(),colorscale='Blues',colorbar=dict(title='Count'),opacity=0.7),
    text=[
        f'{itos[i]}{itos[j]}{itos[k]}: {N[i,j,k].item()}'
        for i, j, k in zip(x.flatten(), y.flatten(), z.flatten())
    ],
    hovertemplate='%{text}<extra></extra>'
)])

fig.update_layout(
    title='3D Character Trigram Counts',
    scene=dict(
        xaxis_title='Character 1',
        yaxis_title='Character 2',
        zaxis_title='Character 3',
    ),
    width=1000,
    height=900
)

fig.show()

In [276]:
#smoothen to avoid 0s, normalize
P = (N+1).float()
P = P/P.sum(2, keepdim=True)

In [286]:
#generate
g = torch.Generator()
for i in range(10):
    ix1 = 0
    ix2 = 0
    out = []
    while True:
        p = P[ix1, ix2]
        ix = torch.multinomial(p, num_samples = 1, replacement = True).item()
        out.append(itos[ix])
        ix1 = ix2
        ix2 = ix
        if(ix==0):
            break
    print(''.join(out))

haddevib.
yanthyae.
gavidevyan.
sakrunmaramriya.
ez.
kaashita.
mriam.
anan.
odivyayeshashthi.
yanthame.


In [287]:
#measure loss (MLE)
log_likelihood = 0.0
n=0
for word in ["yogesh"]:
    chs = ['.'] + list(word) + ['.']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]
        prob = P[ix1, ix2, ix3]
        logprob = torch.log(prob)
        log_likelihood += logprob
        n+=1
        print(f'{ch1}{ch2}{ch3}: {prob:.4f}, {logprob:.4f}')
log_likelihood
nll = -log_likelihood
nll/n

.yo: 0.1673, -1.7882
yog: 0.2753, -1.2898
oge: 0.1123, -2.1863
ges: 0.2331, -1.4565
esh: 0.6958, -0.3627
sh.: 0.1379, -1.9812


tensor(1.5108)